# 🦺 YOLOv8 PPE Detection Training

This notebook trains a custom YOLOv8 model to detect PPE (Personal Protective Equipment) compliance on construction sites.

**What you'll get:**
- Custom trained model (`best.pt`)
- Detects: Hardhat, Safety Vest, No-Hardhat, No-Safety Vest

**Time required:** ~45-60 minutes on free Colab T4 GPU

---

## Step 0: Enable GPU

**Before running anything:**
1. Go to **Runtime** → **Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

In [ ]:
# Verify GPU is enabled
!nvidia-smi

## Step 1: Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q
print("✅ Dependencies installed!")

## Step 2: Download Dataset from Roboflow

⚠️ **Replace `YOUR_API_KEY` with your actual Roboflow API key**

In [ ]:
from roboflow import Roboflow

# ========================================
# 👇 PASTE YOUR API KEY HERE 👇
# ========================================
rf = Roboflow(api_key="YOUR_API_KEY")
# ========================================

project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

## Step 3: Train YOLOv8

Training configuration:
- **Model:** YOLOv8s (small) — good balance of speed and accuracy
- **Epochs:** 50 (can increase for better results)
- **Image size:** 640x640
- **Batch size:** 16 (fits in T4 memory)

☕ This takes **45-60 minutes** on a free T4 GPU. Good time for a break!

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s model
model = YOLO("yolov8s.pt")

# Train on PPE dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="ppe_detector",
    patience=10,  # Early stopping if no improvement
    save=True,
    plots=True
)

print("\n" + "="*50)
print("✅ TRAINING COMPLETE!")
print("="*50)

## Step 4: Evaluate Model Performance

In [ ]:
# Validate on test set
metrics = model.val()

print("\n📊 Model Performance:")
print(f"   mAP50:     {metrics.box.map50:.3f}")
print(f"   mAP50-95:  {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall:    {metrics.box.mr:.3f}")

## Step 5: Test on Sample Images

In [ ]:
import glob
from IPython.display import Image, display

# Load the best trained model
best_model = YOLO("runs/detect/ppe_detector/weights/best.pt")

# Get some test images
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:3]

# Run inference
for img_path in test_images:
    results = best_model(img_path, save=True, project="test_results", name="predictions")

# Display results
print("\n🖼️ Sample Predictions:\n")
for img in glob.glob("test_results/predictions/*.jpg")[:3]:
    display(Image(filename=img, width=500))
    print("---")

## Step 6: Download Your Trained Model

Download `best.pt` — this is what you'll use in the Gradio app.

In [ ]:
from google.colab import files
import shutil

# Copy best model to easy location
shutil.copy("runs/detect/ppe_detector/weights/best.pt", "ppe_detector_best.pt")

# Download it
print("📥 Downloading trained model...")
files.download("ppe_detector_best.pt")

print("\n✅ Done! Upload 'ppe_detector_best.pt' to your Hugging Face Space.")

## 📋 Training Summary

View all training plots and metrics:

In [ ]:
# Display training curves
from IPython.display import Image, display

print("📈 Training Results:\n")
display(Image(filename="runs/detect/ppe_detector/results.png", width=800))

print("\n📊 Confusion Matrix:\n")
display(Image(filename="runs/detect/ppe_detector/confusion_matrix.png", width=600))

---

## ✅ Next Steps

1. **Download** `ppe_detector_best.pt` (should have auto-downloaded above)
2. **Upload** to your Hugging Face Space alongside the new `app.py`
3. **Deploy** and test!

Your model detects:
- ✅ Hardhat
- ✅ Safety Vest  
- ⚠️ NO-Hardhat (violation)
- ⚠️ NO-Safety Vest (violation)